# Extension to 2100 Walkthrough

This notebook is a readable companion to `un_extension_2100.py`.

It keeps the implementation in one Python script, but exposes the longer-horizon workflow step by step:

1. load the extension module
2. validate required inputs
3. load the INE replication module used as the baseline
4. run the `2074-2100` extension
5. inspect the UN target path and the resulting life-table `e0`
6. write the Excel outputs and validation workbook

## How to use this notebook

- The notebook is meant for reading and experimentation.
- The script `un_extension_2100.py` remains the implementation source of truth.
- If you want to regenerate the extension outputs exactly as in the repo workflow, execute all cells in order.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "un_extension_2100.py").exists():
    script_path = cwd / "un_extension_2100.py"
elif (cwd / "code" / "extension_2100" / "un_extension_2100.py").exists():
    script_path = cwd / "code" / "extension_2100" / "un_extension_2100.py"
else:
    raise FileNotFoundError("Could not locate un_extension_2100.py from the current working directory.")

spec = importlib.util.spec_from_file_location("un_extension_2100", script_path)
ue = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(ue)

print(f"Loaded module from: {script_path}")
print(f"Project root: {ue.PROJECT_ROOT}")

In [ ]:
for path in [ue.REPLICATION_SCRIPT, ue.UN_MALE_E0_FILE, ue.UN_FEMALE_E0_FILE]:
    ue.require_file(path)
ue.ensure_dirs()
print("All required extension files were found and output directories are ready.")

## Baseline and extension logic

This stage starts from the validated INE-style replication through `2073` and then extends the horizon to `2100`.

- Up to `2073`, the benchmark is INE.
- From `2074` onward, the benchmark is the annualized UN target path.
- So the extension workbook does not compare `2074-2100` against INE tables.

In [ ]:
replication = ue.load_replication_module()
print(f"Loaded replication module from: {ue.REPLICATION_SCRIPT}")

## Run the extension

This step:

- reuses the validated baseline replication through `2073`
- loads the UN life-expectancy path for Spain
- annualizes the UN quinquennial values
- extends annual `qx`, `ax`, and life-table `e0` to `2100`

In [ ]:
results = ue.run_extension(replication)
list(results.keys())

In [ ]:
pd.DataFrame(
    {
        "male_target_e0": results["male"]["life_extension"].set_index("Year")["target_e0"],
        "male_life_table_e0": results["male"]["life_extension"].set_index("Year")["e0_life_table"],
        "female_target_e0": results["female"]["life_extension"].set_index("Year")["target_e0"],
        "female_life_table_e0": results["female"]["life_extension"].set_index("Year")["e0_life_table"],
    }
).head()

In [ ]:
summary_rows = []
for sex, result in results.items():
    life_extension = result["life_extension"]
    summary_rows.append(
        {
            "sex": sex,
            "mean_abs_error_vs_un_target": life_extension["abs_error_vs_target"].mean(),
            "max_abs_error_vs_un_target": life_extension["abs_error_vs_target"].max(),
        }
    )

pd.DataFrame(summary_rows)

## Annualization note

The extension treats the UN quinquennial values as knots at the period end years `2075, 2080, ..., 2100` and interpolates annual targets between them.

That choice is documented separately in `un_annualization_note.md` because it is a modelling assumption, not a direct output of the UN workbook.

## Write the Excel outputs

These two calls reproduce the normal script behavior:

- write the annual target paths and projected tables to `output/mortality_projection/...`
- write the validation workbook comparing the extension result against the annualized UN target

In [ ]:
ue.write_extension_outputs(results)
ue.write_validation_workbook(results)

print(f"Final output folder: {ue.FINAL_DIR}")
print(f"Validation workbook: {ue.VALIDATION_FILE}")

## Next step

Once this extension stage is considered stable, the next improvements should focus on methodology quality rather than adding extra variants: for example, refining the UN bridge assumption or simplifying the presentation of the final outputs.